In [4]:
from sklearn.linear_model import ridge_regression
from sklearn.model_selection import train_test_split
from skfp.datasets.moleculenet import load_esol
from skfp.model_selection import scaffold_train_test_split

import numpy as np
import seaborn as sns
import pandas as pd

from rdkit import Chem

from rdkit.Chem import rdFingerprintGenerator

In [2]:
def load_dataset() -> pd.DataFrame:
    from skfp.datasets.moleculenet import load_esol
    return load_esol(data_dir='../data', as_frame=True)

df = load_dataset()
df.describe()

,label
count,1128.000000
mean,-3.050102
std,2.096441
min,-11.600000
25%,-4.317500
50%,-2.860000
75%,-1.600000
max,1.580000


In [3]:
# some health checks
def parse_check(df: pd.DataFrame):
    bad = [i for i, s in enumerate(df['SMILES']) if Chem.MolFromSmiles(s) is None]
    print(f"Unparseable SMILES: {len(bad)}")
    return bad

bad = parse_check(df)

def duplicate_check(df: pd.DataFrame):
    canon = []

    for s in df['SMILES']:
        mol = Chem.MolFromSmiles(s)
        canon.append(mol if mol is not None else None)
    canon = pd.Series(canon)
    print(f'Number of duplicates: {canon.duplicated().sum()}')

duplicate_check(df)

def mean_predictor_rmse(y_train: np.ndarray, y_test: np.ndarray) -> float:
    """RMSE of predicting the TRAINING mean for every test example."""
    c = y_train.mean()
    return float(np.sqrt(np.mean((y_test - c) ** 2)))

Unparseable SMILES: 0
Number of duplicates: 0


In [21]:
# featurization
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
mol = Chem.MolFromSmiles("CC(=O)Oc1ccccc1C(=O)O")
fp = gen.GetCountFingerprintAsNumPy(mol)

print(len(fp), fp.sum(), np.count_nonzero(fp))

def tanimoto(a, b):
    return float(np.min(a, b).sum() / np.max(a, b))

RADIUS = 2
N_BITS = 2048

def make_generator(radius, n_bits):
    return rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
gen = make_generator(RADIUS, N_BITS)
fp = gen.GetCountFingerprintAsNumPy(mol)

def featurize(smiles: list[str], generator):
    rows, keep = [], []
    for s in smiles:
        mol = Chem.MolFromSmiles(s)
        keep.append(mol is not None)
        if mol is not None:
            rows.append(generator.GetCountFingerprintAsNumPy(mol))

    rows, keep = np.stack(rows), np.stack(keep)
    return rows, keep

smiles = df['SMILES']
gen = make_generator(RADIUS, N_BITS)
rows, keep = featurize(smiles, gen)

2048 35 24


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.metrics import root_mean_squared_error
df = load_dataset()

smiles = df['SMILES']
generator = make_generator(RADIUS, N_BITS)
X, keep = featurize(smiles, generator)
y = df[keep]['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

model = Ridge()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = root_mean_squared_error(y_pred, y_test)
print(rmse)
# 0.9476982329912494

0.9476982329912494


In [ ]:
control = root_mean_squared_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"CONTROL mean-predictor RMSE: {control:.4f}")

CONTROL mean-predictor RMSE: 2.1743


In [58]:
from sklearn.model_selection import KFold
ALPHAS = (1, 2, 3, 4, 5, 6, 7, 8, 9)

def select_alpha(X, y, alphas=ALPHAS, n_folds=5, seed=0) -> tuple[float, dict]:
    """Choose alpha by K-fold CV on the TRAINING set only. Never touches test."""
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    scores = {}
    for a in alphas:
        fold = []
        for tr, va in kf.split(X):
            model = Ridge(alpha=a).fit(X[tr], y[tr])
            fold.append(root_mean_squared_error(y[va], model.predict(X[va])))
        scores[a] = float(np.mean(fold))
    best = min(scores, key=scores.get)
    return best, scores

In [ ]:
df = load_dataset()
gen = make_generator(RADIUS, N_BITS)

X_all, keep = featurize(df["SMILES"].tolist(), gen)
y_all = df["label"].to_numpy(dtype=np.float64)[keep]
print(f"molecules kept: {len(y_all)}  dropped: {int((~keep).sum())}")
print(f"feature matrix X: {X_all.shape}")
print(f"mean non-zero features per molecule: {np.mean((X_all > 0).sum(axis=1)):.1f}")
print(f"columns zero for every molecule: {int((X_all.sum(axis=0) == 0).sum())}")

# Random 80/20 split, same seed as day 3 so the control number matches.
rng = np.random.default_rng(0)
idx = rng.permutation(len(y_all))
cut = int(0.8 * len(y_all))
tr, te = idx[:cut], idx[cut:]
X_tr, y_tr, X_te, y_te = X_all[tr], y_all[tr], X_all[te], y_all[te]

control = root_mean_squared_error(y_te, np.full_like(y_te, y_tr.mean()))
print(f"CONTROL mean-predictor RMSE: {control:.4f}")

best_alpha, cv_scores = select_alpha(X_tr, y_tr)
for a, s in cv_scores.items():
    mark = "  <- chosen" if a == best_alpha else ""
    print(f"  alpha={a:>8.2f}   cv_rmse={s:.4f}{mark}")

model = Ridge(alpha=best_alpha).fit(X_tr, y_tr)
pred = model.predict(X_te)
res = root_mean_squared_error(y_te, pred)


molecules kept: 1128  dropped: 0
feature matrix X: (1128, 2048)
mean non-zero features per molecule: 21.5
columns zero for every molecule: 192
CONTROL mean-predictor RMSE: 2.2083
  alpha=    1.00   cv_rmse=0.9284
  alpha=    2.00   cv_rmse=0.8961
  alpha=    3.00   cv_rmse=0.8806
  alpha=    4.00   cv_rmse=0.8716
  alpha=    5.00   cv_rmse=0.8660
  alpha=    6.00   cv_rmse=0.8624
  alpha=    7.00   cv_rmse=0.8600
  alpha=    8.00   cv_rmse=0.8585
  alpha=    9.00   cv_rmse=0.8576  <- chosen


In [ ]:
df = load_dataset()
gen = make_generator(RADIUS, N_BITS)

X, y = df['SMILES'].values, df['label'].values
X_train, X_test, y_train, y_test = scaffold_train_test_split(X, y, test_size=0.2)
(X_train, _ ), (X_test, _ ) = featurize(X_train, generator), featurize(X_test, generator)

control = root_mean_squared_error(y_test, np.full_like(y_test, y_train.mean()))
print(f"CONTROL mean-predictor RMSE: {control:.4f}")

best_alpha, cv_scores = select_alpha(X_train, y_train)
for a, s in cv_scores.items():
    mark = "  <- chosen" if a == best_alpha else ""
    print(f"  alpha={a:>8.2f}   cv_rmse={s:.4f}{mark}")

model = Ridge(alpha=best_alpha).fit(X_train, y_train)
pred = model.predict(X_test)
res = root_mean_squared_error(y_test, pred)
print(res)
# 1.2345824958396592

CONTROL mean-predictor RMSE: 2.2443
  alpha=    1.00   cv_rmse=0.7997
  alpha=    2.00   cv_rmse=0.7674
  alpha=    3.00   cv_rmse=0.7548
  alpha=    4.00   cv_rmse=0.7494
  alpha=    5.00   cv_rmse=0.7474
  alpha=    6.00   cv_rmse=0.7473  <- chosen
  alpha=    7.00   cv_rmse=0.7483
  alpha=    8.00   cv_rmse=0.7499
  alpha=    9.00   cv_rmse=0.7520
1.2345824958396592
